# Programming Assignment 01 — Version 2 (LOCAL) — Gradio Tabs: Web Unlocker Scraper + YouTube Transcript Q&A

This notebook implements **Version 2** end-to-end **for local execution** (no Hugging Face Spaces).

✅ **Tab 1:** Bot-protected Website Scraper (**Bright Data Web Unlocker API** + BeautifulSoup) + Contextual Q&A  
✅ **Tab 2:** YouTube Transcript fetch (**youtube-transcript-api**) + Contextual Q&A  
✅ Runs locally via `gradio.Blocks()`.

---

## What you need before running

### 1) Bright Data (required for Tab 1)
Set these environment variables:

- `BRIGHTDATA_API_KEY` → your Bright Data API key (Bearer token)
- `BRIGHTDATA_UNLOCKER_ZONE` → your Web Unlocker **zone name**

Bright Data direct API endpoint used: `POST https://api.brightdata.com/request` with `format="raw"` to receive raw HTML.

### 2) LLM Provider (choose one)
You can run either:

- **Local (recommended for “local assignment”)**: Ollama (`http://localhost:11434/v1`)
- **Groq (optional)**: set `GROQ_API_KEY` and use Groq’s OpenAI-compatible endpoint (`https://api.groq.com/openai/v1`)

> If `GROQ_API_KEY` is not set, the notebook automatically uses Ollama.

### 3) YouTube transcript
No key needed for transcript fetching. Some videos have transcripts disabled; the app shows a clear error.

---

## Quick run order
1. Install deps  
2. Configure env vars  
3. Run the app cell (Gradio UI opens locally)


In [ ]:
# If you're running inside a fresh venv, install deps:
# (Run this cell once)
!pip -q install -U gradio openai python-dotenv requests beautifulsoup4 pandas youtube-transcript-api lxml


## Environment variables

### Option A: Export in terminal before launching Jupyter
```bash
export BRIGHTDATA_API_KEY="YOUR_BRIGHTDATA_API_KEY"
export BRIGHTDATA_UNLOCKER_ZONE="YOUR_UNLOCKER_ZONE_NAME"

# Optional (only if you want Groq instead of Ollama):
export GROQ_API_KEY="YOUR_GROQ_API_KEY"
```

### Option B: Use a local `.env` file (same folder as this notebook)
Create a file named `.env`:
```env
BRIGHTDATA_API_KEY=...
BRIGHTDATA_UNLOCKER_ZONE=...
GROQ_API_KEY=...   # optional
```


## Ollama (Local LLM) setup (recommended)

1) Install Ollama  
2) Pull a model (example):
```bash
ollama pull llama3.2
```

Ollama provides an OpenAI-compatible endpoint at `http://localhost:11434/v1`.


In [1]:

import os
import re
import textwrap
from typing import Tuple, Optional, List, Dict

import requests
import pandas as pd
from bs4 import BeautifulSoup
import gradio as gr

from dotenv import load_dotenv
from openai import OpenAI

# ----------------------------
# 0) Load .env (optional)
# ----------------------------
# NOTE:
# - In Jupyter/VS Code, environment variables are captured when the kernel starts.
#   If you export variables in a different terminal *after* the kernel is running,
#   this process won't see them until you restart the kernel (or set os.environ in Python).
load_dotenv(override=True)

def _getenv(name: str, default: str = "") -> str:
    return (os.getenv(name, default) or "").strip()

def get_brightdata_creds():
    """Return (api_key, zone). Accepts a few common variable-name variants."""
    api_key = _getenv("BRIGHTDATA_API_KEY") or _getenv("BRIGHT_DATA_API_KEY")
    zone = (
        _getenv("BRIGHTDATA_UNLOCKER_ZONE")
        or _getenv("BRIGHTDATA_ZONE")
        or _getenv("BRIGHTDATA_WEBUNLOCKER_ZONE")
    )
    return api_key, zone

def get_groq_key() -> str:
    return _getenv("GROQ_API_KEY")
# ----------------------------
# 1) LLM Provider selection
# ----------------------------
def make_llm_client(provider: str) -> OpenAI:
    # provider: "ollama" (local) or "groq" (cloud)
    provider = (provider or "").lower().strip()
    if provider == "groq":
        if not get_groq_key():
            raise RuntimeError("GROQ_API_KEY is not set. Either set it or choose provider='ollama'.")
        return OpenAI(base_url="https://api.groq.com/openai/v1", api_key=get_groq_key())
    # default = ollama
    return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

DEFAULT_PROVIDER = "groq" if get_groq_key() else "ollama"
DEFAULT_MODEL = "llama-3.1-8b-instant" if DEFAULT_PROVIDER == "groq" else "llama3.2"

# ----------------------------
# 2) Bright Data Web Unlocker (Direct API access)
# ----------------------------
BRIGHTDATA_ENDPOINT = "https://api.brightdata.com/request"

def brightdata_fetch_raw_html(url: str, timeout: int = 120, api_key: str = "", zone: str = "") -> str:
    # Fetch raw HTML via Bright Data Web Unlocker API (direct API access).
    # You can provide api_key/zone explicitly (UI), or via env vars.
    api_key = (api_key or "").strip()
    zone = (zone or "").strip()
    if not api_key or not zone:
        api_key_env, zone_env = get_brightdata_creds()
        api_key = api_key or api_key_env
        zone = zone or zone_env
    if not api_key or not zone:
        raise RuntimeError(
            "Bright Data credentials are not visible to this Python process.\n"
            "Fix options:\n"
            "1) Restart the Jupyter kernel after exporting env vars, OR\n"
            "2) Put a .env file next to this notebook, OR\n"
            "3) Paste creds into the UI accordion.\n\n"
            "Expected env vars: BRIGHTDATA_API_KEY and BRIGHTDATA_UNLOCKER_ZONE (or BRIGHTDATA_ZONE)."
        )

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    payload = {
        "zone": zone,
        "url": url,
        "format": "raw",   # raw => return target site's raw body (HTML)
        "method": "GET",
    }

    resp = requests.post(BRIGHTDATA_ENDPOINT, headers=headers, json=payload, timeout=timeout)
    if resp.status_code != 200:
        raise RuntimeError(f"Bright Data request failed ({resp.status_code}): {resp.text[:500]}")
    return resp.text

# ----------------------------
# 3) HTML parsing helpers
# ----------------------------
def extract_goodreads_books(html: str, max_items: int = 30) -> pd.DataFrame:
    # Parse Goodreads list pages: Book Title, Author Name, Star Rating (avg rating).
    # Returns empty DataFrame if not detected.
    soup = BeautifulSoup(html, "lxml")
    rows = soup.select("table.tableList tr")
    books = []
    for r in rows:
        title_el = r.select_one("a.bookTitle span") or r.select_one("a.bookTitle")
        author_el = r.select_one("a.authorName span") or r.select_one("a.authorName")
        rating_el = r.select_one("span.minirating") or r.select_one("span.greyText.smallText")

        if not title_el or not author_el:
            continue

        title = title_el.get_text(" ", strip=True)
        author = author_el.get_text(" ", strip=True)

        rating = None
        if rating_el:
            txt = rating_el.get_text(" ", strip=True)
            m = re.search(r"([0-9]\\.[0-9]{1,2})\\s+avg rating", txt) or re.search(r"([0-9]\\.[0-9]{1,2})", txt)
            if m:
                try:
                    rating = float(m.group(1))
                except:
                    rating = None

        books.append({"Book Title": title, "Author Name": author, "Star Rating": rating})
        if len(books) >= max_items:
            break

    return pd.DataFrame(books)

def extract_visible_text(html: str, max_chars: int = 15000) -> str:
    # Generic fallback: strip scripts/styles and return visible body text.
    soup = BeautifulSoup(html, "lxml")
    body = soup.body or soup
    for tag in body(["script", "style", "noscript", "svg", "img", "input"]):
        tag.decompose()
    text = body.get_text(separator="\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text[:max_chars]

def build_scrape_context(url: str, html: str) -> Tuple[str, pd.DataFrame, str]:
    # Returns: (context_text, df, preview)
    df = extract_goodreads_books(html)
    if not df.empty:
        context = "Scraped Goodreads data (Book Title, Author Name, Star Rating):\n" + df.to_json(orient="records", indent=2)
        preview = df.head(10).to_string(index=False)
        return context, df, preview

    text = extract_visible_text(html, max_chars=15000)
    context = f"Scraped webpage text from: {url}\n\n{text}"
    preview = textwrap.shorten(text, width=800, placeholder=" ...")
    return context, pd.DataFrame(), preview

# ----------------------------
# 4) YouTube transcript fetch
# ----------------------------
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable, CouldNotRetrieveTranscript

def normalize_video_id(video_id_or_url: str) -> str:
    s = (video_id_or_url or "").strip()
    if not s:
        return ""
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", s):
        return s
    m = re.search(r"[?&]v=([A-Za-z0-9_-]{11})", s)
    if m:
        return m.group(1)
    m = re.search(r"youtu\\.be/([A-Za-z0-9_-]{11})", s)
    if m:
        return m.group(1)
    m = re.search(r"/shorts/([A-Za-z0-9_-]{11})", s)
    if m:
        return m.group(1)
    return s

def fetch_transcript_text(video_id: str, prefer_langs: Optional[List[str]] = None) -> str:
    # Robust transcript fetch:
    # 1) Try the stable convenience method get_transcript()
    # 2) If that fails, fall back to list()+find_transcript() (useful when language selection is tricky)
    prefer_langs = prefer_langs or ["en"]

    # Attempt 1: direct fetch
    try:
        data = YouTubeTranscriptApi.get_transcript(video_id, languages=prefer_langs)
        return " ".join([t.get("text", "") for t in data])
    except Exception:
        pass

    # Attempt 2: list + select
    api = YouTubeTranscriptApi()
    transcript_list = api.list(video_id)

    transcript = None
    for lang in prefer_langs:
        try:
            transcript = transcript_list.find_transcript([lang])
            break
        except Exception:
            continue

    if transcript is None:
        # Pick the first available transcript (manual preferred)
        available = list(transcript_list._manually_created_transcripts.keys()) + list(transcript_list._generated_transcripts.keys())
        if not available:
            raise NoTranscriptFound(video_id, prefer_langs, [])
        transcript = transcript_list.find_transcript([available[0]])

    data = transcript.fetch()
    parts = []
    for t in data:
        if isinstance(t, dict):
            parts.append(t.get("text", ""))
        else:
            parts.append(getattr(t, "text", ""))
    return " ".join(parts)

# ----------------------------
# 5) LLM Q&A helper
# ----------------------------
def qa_with_context(provider: str, model: str, context: str, question: str, temperature: float = 0.2) -> str:
    client = make_llm_client(provider)

    system_prompt = (
        "You are a helpful assistant that answers questions based ONLY on the provided CONTEXT.\n"
        "Rules:\n"
        "1) Use ONLY information explicitly present in CONTEXT.\n"
        "2) If the answer is not in CONTEXT, say: "
        "\"I cannot answer this question because the information is not available in the provided context.\"\n"
        "3) Do NOT use outside knowledge.\n"
        "4) Be concise and specific.\n"
    )

    context_trim = (context or "")[:15000]
    user_prompt = f"CONTEXT:\n{context_trim}\n\nQUESTION:\n{question}\n"

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content

# ----------------------------
# 6) Gradio App (Version 2)
# ----------------------------
def load_url(url: str, bd_api_key: str, bd_zone: str):
    url = (url or "").strip()
    if not url:
        return "❌ Please enter a URL.", "", pd.DataFrame(), ""
    try:
        html = brightdata_fetch_raw_html(url, api_key=bd_api_key, zone=bd_zone)
        context, df, preview = build_scrape_context(url, html)
        return "✅ Loaded and parsed successfully.", context, df, preview
    except Exception as e:
        return f"❌ Error: {e}", "", pd.DataFrame(), ""

def chat_web(message: str, history: List[Dict], context: str, provider: str, model: str):
    if not context:
        return "Please click **Load URL** first to fetch & parse content."
    try:
        return qa_with_context(provider, model, context, message)
    except Exception as e:
        return f"❌ LLM error: {e}"

def load_video(video_id_or_url: str, lang: str):
    vid = normalize_video_id(video_id_or_url)
    if not vid:
        return "❌ Please enter a YouTube Video ID (or URL).", "", ""
    try:
        transcript_text = fetch_transcript_text(vid, prefer_langs=[lang] if lang else ["en"])
        context = f"YouTube transcript for video_id={vid}\n\n{transcript_text}"
        preview = textwrap.shorten(transcript_text, width=1200, placeholder=" ...")
        return "✅ Transcript loaded successfully.", context, preview
    except (TranscriptsDisabled, NoTranscriptFound, CouldNotRetrieveTranscript) as e:
        return f"❌ No transcript available for this video ({vid}). ({type(e).__name__})", "", ""
    except VideoUnavailable as e:
        return f"❌ Video unavailable ({vid}). ({type(e).__name__})", "", ""
    except Exception as e:
        return f"❌ Error: {e}", "", ""

def chat_youtube(message: str, history: List[Dict], transcript_context: str, provider: str, model: str):
    if not transcript_context:
        return "Please click **Load Transcript** first."
    try:
        return qa_with_context(provider, model, transcript_context, message)
    except Exception as e:
        return f"❌ LLM error: {e}"

with gr.Blocks(title="MSDSF — Assignment Ver 2 (Local)") as demo:
    gr.Markdown("# Programming Assignment 01 — Version 2 (LOCAL)\n### Tabs: Web Unlocker Scraper + YouTube Transcript Q&A")

    with gr.Row():
        provider = gr.Dropdown(
            choices=["ollama", "groq"],
            value=DEFAULT_PROVIDER,
            label="LLM Provider",
            info="Choose 'ollama' for local, or 'groq' if GROQ_API_KEY is set."
        )
        model = gr.Textbox(
            value=DEFAULT_MODEL,
            label="Model",
            info="Example (Ollama): llama3.2 | Example (Groq): llama-3.1-8b-instant"
        )

    with gr.Tabs():
        with gr.Tab("Tab 1 — Bot-Protected Website Scraper"):
            gr.Markdown("### 1) Enter a URL → Load & Parse → Ask questions in chat")

            url_in = gr.Textbox(label="Target URL", placeholder="https://www.goodreads.com/list/show/1.Best_Books_Ever", lines=1)
            with gr.Accordion("Bright Data credentials (optional)", open=False):
                bd_api_key_in = gr.Textbox(label="BRIGHTDATA_API_KEY", type="password", placeholder="Leave empty to use env var")
                bd_zone_in = gr.Textbox(label="BRIGHTDATA_UNLOCKER_ZONE", placeholder="e.g., msdsf25m012_webunlocker (or leave empty to use env var)")
            load_btn = gr.Button("Load URL", variant="primary")

            status1 = gr.Markdown()
            df_out = gr.Dataframe(label="Extracted Table (if detected)", interactive=False)
            preview1 = gr.Textbox(label="Preview", lines=10)

            context_state1 = gr.State(value="")

            load_btn.click(fn=load_url, inputs=[url_in, bd_api_key_in, bd_zone_in], outputs=[status1, context_state1, df_out, preview1])

            gr.Markdown("### Q&A Chat (uses parsed content as context)")
            gr.ChatInterface(
                fn=chat_web,
                additional_inputs=[context_state1, provider, model],
                chatbot=gr.Chatbot(height=380),
                textbox=gr.Textbox(placeholder="Ask about the scraped content...", container=False),
                title="Website Q&A",
                description="Ask questions grounded ONLY in the scraped content.",
            )

        with gr.Tab("Tab 2 — YouTube Transcript Q&A"):
            gr.Markdown("### 2) Enter a YouTube Video ID → Load Transcript → Ask questions in chat")

            video_in = gr.Textbox(label="YouTube Video ID (or URL)", placeholder="dQw4w9WgXcQ", lines=1)
            lang_in = gr.Dropdown(choices=["en", "ur", "hi", "es", "fr", "de"], value="en", label="Preferred transcript language")
            load_vid_btn = gr.Button("Load Transcript", variant="primary")

            status2 = gr.Markdown()
            preview2 = gr.Textbox(label="Transcript preview", lines=12)

            context_state2 = gr.State(value="")

            load_vid_btn.click(fn=load_video, inputs=[video_in, lang_in], outputs=[status2, context_state2, preview2])

            gr.Markdown("### Q&A Chat (uses transcript as context)")
            gr.ChatInterface(
                fn=chat_youtube,
                additional_inputs=[context_state2, provider, model],
                chatbot=gr.Chatbot(height=380),
                textbox=gr.Textbox(placeholder="Ask about the video transcript...", container=False),
                title="YouTube Transcript Q&A",
                description="Ask questions grounded ONLY in the transcript.",
            )

    gr.Markdown("✅ Done. This is Version 2 running locally (no Hugging Face deployment).")

demo


Gradio Blocks instance: 62 backend functions
--------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x76643c30f050>
 |-<gradio.components.textbox.Textbox object at 0x76643c2a67e0>
 |-<gradio.components.textbox.Textbox object at 0x76643c2d2f90>
 outputs:
 |-<gradio.components.markdown.Markdown object at 0x76643cf33fe0>
 |-<gradio.components.state.State object at 0x76643c228980>
 |-<gradio.components.dataframe.Dataframe object at 0x76643c2a5250>
 |-<gradio.components.textbox.Textbox object at 0x76643c229670>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x76643c31cec0>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x76643c31cec0>
 |-<gradio.components.state.State object at 0x76643c15a810>
fn_index=2
 inputs:
 |-<gradio.components.state.State object at 0x76643c15a810>
 |-<gradio.components.chatbot.Chatbot object at 0x76643c2294f0>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x76643c22

In [2]:
# Run the Gradio app locally
# - share=False (local only)
# - inbrowser=True (opens in browser)
demo.launch(share=False, inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
